# Cost-Sensitive Threshold Optimization
named_steps
scaler
This notebook sweeps thresholds to minimize business cost and saves optimal metrics to models/metadata.json.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score

In [ ]:
BASE_FEATURES = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
raw_path = Path("data/creditcard.csv")
proc_path = Path("data/processed.csv")

if proc_path.exists():
    df = pd.read_csv(proc_path)
elif raw_path.exists():
    df = pd.read_csv(raw_path)
else:
    raise FileNotFoundError("Missing data file. Place creditcard.csv in data/ or run notebooks/02_features.py.")

required = BASE_FEATURES + ["Class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required]
X = df[BASE_FEATURES]
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

models_dir = Path("models")
model_path = models_dir / "fraud_model.pkl"
scaler_path = models_dir / "scaler.pkl"

if not model_path.exists():
    raise FileNotFoundError("Missing models/fraud_model.pkl. Run the training notebook first.")

model = joblib.load(model_path)
scaler = joblib.load(scaler_path) if scaler_path.exists() else None

X_test_in = scaler.transform(X_test) if scaler is not None else X_test.values
proba = model.predict_proba(X_test_in)[:, 1]
roc_auc = roc_auc_score(y_test, proba)
print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
COST_FALSE_POSITIVE = 50
COST_FALSE_NEGATIVE = 500

def compute_cost(y_true, y_pred, cost_fp, cost_fn):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fp * cost_fp + fn * cost_fn, (tn, fp, fn, tp)

thresholds = np.linspace(0.01, 0.99, 99)
costs = []
detections = []
false_alarms = []

for t in thresholds:
    preds = (proba >= t).astype(int)
    cost, (tn, fp, fn, tp) = compute_cost(y_test, preds, COST_FALSE_POSITIVE, COST_FALSE_NEGATIVE)
    costs.append(cost)
    detection = tp / (tp + fn) if (tp + fn) > 0 else 0
    false_alarm = fp / (fp + tn) if (fp + tn) > 0 else 0
    detections.append(detection)
    false_alarms.append(false_alarm)

costs = np.array(costs)
opt_idx = int(costs.argmin())
opt_t = float(thresholds[opt_idx])
opt_cost = float(costs[opt_idx])
opt_detection = float(detections[opt_idx])
opt_false_alarm = float(false_alarms[opt_idx])

print(f"Optimal threshold: {opt_t:.3f}")
print(f"Minimum cost: ${opt_cost:,.0f}")
print(f"Fraud detection rate: {opt_detection:.2%}")
print(f"False alarm rate: {opt_false_alarm:.2%}")

default_preds = (proba >= 0.5).astype(int)
default_cost, _ = compute_cost(y_test, default_preds, COST_FALSE_POSITIVE, COST_FALSE_NEGATIVE)
cost_savings_percent = ((default_cost - opt_cost) / default_cost) * 100 if default_cost > 0 else 0.0
print(f"Cost savings vs 0.5 threshold: {cost_savings_percent:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(thresholds, costs, marker="o", linewidth=1)
axes[0].axvline(opt_t, color="red", linestyle="--", label=f"Optimal {opt_t:.2f}")
axes[0].axvline(0.5, color="gray", linestyle="--", label="Default 0.50")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Total cost")
axes[0].set_title("Cost vs Threshold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, detections, label="Fraud detection", marker="o")
axes[1].plot(thresholds, false_alarms, label="False alarm", marker="s")
axes[1].axvline(opt_t, color="red", linestyle="--", alpha=0.6)
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Rate")
axes[1].set_title("Detection vs False Alarms")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
opt_preds = (proba >= opt_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, opt_preds).ravel()

metadata = {
    "optimal_threshold": float(opt_t),
    "optimal_cost": float(opt_cost),
    "roc_auc": float(roc_auc),
    "fraud_detection_rate": float(opt_detection),
    "false_alarm_rate": float(opt_false_alarm),
    "cost_savings_percent": float(cost_savings_percent),
    "confusion": {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn)
    },
    "cost_false_positive": float(COST_FALSE_POSITIVE),
    "cost_false_negative": float(COST_FALSE_NEGATIVE),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "features": BASE_FEATURES
}

models_dir.mkdir(parents=True, exist_ok=True)
with open(models_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata to models/metadata.json")